In [6]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from collections import Counter

## Credit Card

In [7]:
classification_reports = []
classification_report_keys = []

In [8]:
random_state = 42

### Data

In [9]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


### Undersampling

In [10]:
X = df.drop(columns='Class')
y = df['Class']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 284315, 1: 492})
Resampled dataset shape: Counter({0: 492, 1: 492})


### Split Data

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

#### Linear Regression

In [12]:
model = LogisticRegression()

In [13]:
model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [14]:
y_pred = model.predict(X_test)

In [15]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logistic Regression')

              precision    recall  f1-score   support

           0       0.96      0.93      0.94        99
           1       0.93      0.96      0.94        98

    accuracy                           0.94       197
   macro avg       0.94      0.94      0.94       197
weighted avg       0.94      0.94      0.94       197



#### Ridge

In [16]:
from sklearn.linear_model import RidgeClassifier


ridge_model = RidgeClassifier(random_state=random_state)

In [17]:
ridge_model.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [18]:
y_pred = ridge_model.predict(X_test)

In [19]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Ridge Classifier')


              precision    recall  f1-score   support

           0       0.89      0.99      0.94        99
           1       0.99      0.88      0.93        98

    accuracy                           0.93       197
   macro avg       0.94      0.93      0.93       197
weighted avg       0.94      0.93      0.93       197



#### Lasso

In [20]:
lasso_model = LogisticRegression(penalty='l1', solver='liblinear')

In [21]:
lasso_model.fit(X_train, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [22]:
y_pred = lasso_model.predict(X_test)

In [23]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline L1 (Lasso) Logistic Regression')

              precision    recall  f1-score   support

           0       0.95      0.95      0.95        99
           1       0.95      0.95      0.95        98

    accuracy                           0.95       197
   macro avg       0.95      0.95      0.95       197
weighted avg       0.95      0.95      0.95       197



### Scaling Data

In [24]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Parameter Tuning

#### Logistic Regression

In [25]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [26]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

In [27]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_recall_optuna_results)

[I 2026-06-24 19:35:38,615] A new study created in memory with name: no-name-646f10d6-ab43-41b3-bcc2-af4e429bd6e9
[I 2026-06-24 19:35:40,053] Trial 0 finished with value: 0.8907822135670237 and parameters: {'C': 0.8355770815861004, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8907822135670237.
[I 2026-06-24 19:35:41,086] Trial 1 finished with value: 0.8857189224277832 and parameters: {'C': 0.32324092459265447, 'Class_weight': None}. Best is trial 0 with value: 0.8907822135670237.
[I 2026-06-24 19:35:42,144] Trial 2 finished with value: 0.8907822135670237 and parameters: {'C': 0.6654281664652126, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8907822135670237.
[I 2026-06-24 19:35:43,191] Trial 3 finished with value: 0.8907822135670237 and parameters: {'C': 0.8136240165337804, 'Class_weight': None}. Best is trial 0 with value: 0.8907822135670237.
[I 2026-06-24 19:35:43,207] Trial 4 finished with value: 0.8907822135670237 and parameters: {'C': 0.9855053004116283,

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
35,35,0.893314,2026-06-24 19:35:43.722395,2026-06-24 19:35:43.740900,0 days 00:00:00.018505,0.995630,balanced,COMPLETE
24,24,0.893314,2026-06-24 19:35:43.526984,2026-06-24 19:35:43.542928,0 days 00:00:00.015944,0.998962,balanced,COMPLETE
11,11,0.893314,2026-06-24 19:35:43.302127,2026-06-24 19:35:43.318210,0 days 00:00:00.016083,0.998227,balanced,COMPLETE
10,10,0.893314,2026-06-24 19:35:43.286349,2026-06-24 19:35:43.302127,0 days 00:00:00.015778,0.988119,balanced,COMPLETE
31,31,0.893314,2026-06-24 19:35:43.641905,2026-06-24 19:35:43.666723,0 days 00:00:00.024818,0.989176,balanced,COMPLETE


In [28]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_f1_optuna_results)

[I 2026-06-24 19:35:44,885] A new study created in memory with name: no-name-7b559d8b-9980-4dde-80f4-6cc483bcd45e
[I 2026-06-24 19:35:44,913] Trial 0 finished with value: 0.9309924060669379 and parameters: {'C': 0.9836845852940687, 'Class_weight': None}. Best is trial 0 with value: 0.9309924060669379.
[I 2026-06-24 19:35:44,929] Trial 1 finished with value: 0.926405627137804 and parameters: {'C': 0.21005433861759773, 'Class_weight': None}. Best is trial 0 with value: 0.9309924060669379.
[I 2026-06-24 19:35:44,945] Trial 2 finished with value: 0.9267929484786859 and parameters: {'C': 0.32954290205589837, 'Class_weight': None}. Best is trial 0 with value: 0.9309924060669379.
[I 2026-06-24 19:35:44,972] Trial 3 finished with value: 0.9295807643799391 and parameters: {'C': 0.7421361264639691, 'Class_weight': None}. Best is trial 0 with value: 0.9309924060669379.
[I 2026-06-24 19:35:44,993] Trial 4 finished with value: 0.9295807643799391 and parameters: {'C': 0.7178866304090076, 'Class_weig

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.930992,2026-06-24 19:35:44.886714,2026-06-24 19:35:44.913936,0 days 00:00:00.027222,0.983685,None,COMPLETE
15,15,0.930992,2026-06-24 19:35:45.173791,2026-06-24 19:35:45.189125,0 days 00:00:00.015334,0.996624,None,COMPLETE
12,12,0.930992,2026-06-24 19:35:45.124552,2026-06-24 19:35:45.140093,0 days 00:00:00.015541,0.982199,None,COMPLETE
11,11,0.930992,2026-06-24 19:35:45.108752,2026-06-24 19:35:45.124552,0 days 00:00:00.015800,0.999515,None,COMPLETE
31,31,0.930992,2026-06-24 19:35:45.440935,2026-06-24 19:35:45.466208,0 days 00:00:00.025273,0.988656,None,COMPLETE


##### Tryout best params

In [29]:
log_reg_recall_optimized_model = LogisticRegression(C=0.769946, class_weight='balanced', random_state=random_state)
log_reg_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.95      0.98      0.97        99
           1       0.98      0.95      0.96        98

    accuracy                           0.96       197
   macro avg       0.96      0.96      0.96       197
weighted avg       0.96      0.96      0.96       197



In [30]:
log_reg_f1_optimized_model = LogisticRegression(C=0.315005, class_weight='balanced', random_state=random_state)
log_reg_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.95      0.98      0.97        99
           1       0.98      0.95      0.96        98

    accuracy                           0.96       197
   macro avg       0.96      0.96      0.96       197
weighted avg       0.96      0.96      0.96       197



#### Ridge

In [31]:
# We can tune alpha and class weight

##### Recall optimized

In [32]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_recall_optuna_results)

[I 2026-06-24 19:35:46,762] A new study created in memory with name: no-name-1cd54852-5505-4051-92b8-1df93b950d0b
[I 2026-06-24 19:35:46,789] Trial 0 finished with value: 0.8324894514767933 and parameters: {'alpha': 0.838181028632859, 'class_weight': None}. Best is trial 0 with value: 0.8324894514767933.
[I 2026-06-24 19:35:46,805] Trial 1 finished with value: 0.8324894514767933 and parameters: {'alpha': 0.7195146637736881, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8324894514767933.
[I 2026-06-24 19:35:46,821] Trial 2 finished with value: 0.8350210970464136 and parameters: {'alpha': 0.4453398627247352, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.8350210970464136.
[I 2026-06-24 19:35:46,836] Trial 3 finished with value: 0.8324894514767933 and parameters: {'alpha': 0.6956312528965283, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.8350210970464136.
[I 2026-06-24 19:35:46,850] Trial 4 finished with value: 0.8350210970464136 and parameters: {'alp

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
2,2,0.835021,2026-06-24 19:35:46.805010,2026-06-24 19:35:46.821090,0 days 00:00:00.016080,0.445340,balanced,COMPLETE
6,6,0.835021,2026-06-24 19:35:46.868502,2026-06-24 19:35:46.883234,0 days 00:00:00.014732,0.398981,balanced,COMPLETE
5,5,0.835021,2026-06-24 19:35:46.850603,2026-06-24 19:35:46.867999,0 days 00:00:00.017396,0.474322,None,COMPLETE
4,4,0.835021,2026-06-24 19:35:46.836310,2026-06-24 19:35:46.850603,0 days 00:00:00.014293,0.561222,None,COMPLETE
33,33,0.835021,2026-06-24 19:35:47.307807,2026-06-24 19:35:47.322952,0 days 00:00:00.015145,0.414972,balanced,COMPLETE


##### Tryout best params

In [33]:
ridge_recall_optimized_model = RidgeClassifier(alpha=0.039558, class_weight='balanced', random_state=random_state)
ridge_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.89      0.99      0.94        99
           1       0.99      0.88      0.93        98

    accuracy                           0.93       197
   macro avg       0.94      0.93      0.93       197
weighted avg       0.94      0.93      0.93       197



##### F1 optimized

In [34]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_f1_optuna_results)

[I 2026-06-24 19:35:48,437] A new study created in memory with name: no-name-bd43d1ec-76a2-42d5-9b1b-d5478ef749a6
[I 2026-06-24 19:35:48,451] Trial 0 finished with value: 0.8968032437415377 and parameters: {'alpha': 0.6887128400543996, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8968032437415377.
[I 2026-06-24 19:35:48,466] Trial 1 finished with value: 0.8982690274059969 and parameters: {'alpha': 0.3836215244809348, 'class_weight': None}. Best is trial 1 with value: 0.8982690274059969.
[I 2026-06-24 19:35:48,481] Trial 2 finished with value: 0.8982690274059969 and parameters: {'alpha': 0.4879508585697073, 'class_weight': None}. Best is trial 1 with value: 0.8982690274059969.
[I 2026-06-24 19:35:48,497] Trial 3 finished with value: 0.8982690274059969 and parameters: {'alpha': 0.34050341394504147, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8982690274059969.
[I 2026-06-24 19:35:48,519] Trial 4 finished with value: 0.8982690274059969 and parameters: {'alpha':

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
1,1,0.898269,2026-06-24 19:35:48.451034,2026-06-24 19:35:48.466025,0 days 00:00:00.014991,0.383622,None,COMPLETE
2,2,0.898269,2026-06-24 19:35:48.466025,2026-06-24 19:35:48.481757,0 days 00:00:00.015732,0.487951,None,COMPLETE
3,3,0.898269,2026-06-24 19:35:48.481757,2026-06-24 19:35:48.497830,0 days 00:00:00.016073,0.340503,balanced,COMPLETE
4,4,0.898269,2026-06-24 19:35:48.497830,2026-06-24 19:35:48.519195,0 days 00:00:00.021365,0.295365,balanced,COMPLETE
35,35,0.898269,2026-06-24 19:35:48.987945,2026-06-24 19:35:49.004006,0 days 00:00:00.016061,0.104951,balanced,COMPLETE


##### Tryout best params

In [35]:
ridge_f1_optimized_model = RidgeClassifier(alpha=0.364826, class_weight='balanced', random_state=random_state)
ridge_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.89      0.99      0.94        99
           1       0.99      0.88      0.93        98

    accuracy                           0.93       197
   macro avg       0.94      0.93      0.93       197
weighted avg       0.94      0.93      0.93       197



#### Lasso Logistic Regression

In [36]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [37]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, n_jobs=-1)

lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_recall_optuna_results)

[I 2026-06-24 19:35:50,102] A new study created in memory with name: no-name-9feea818-dd95-48bc-8816-891772453b65
[I 2026-06-24 19:35:50,170] Trial 2 finished with value: 0.8857189224277832 and parameters: {'C': 0.6518758124587234, 'Class_weight': None}. Best is trial 2 with value: 0.8857189224277832.
[I 2026-06-24 19:35:50,171] Trial 1 finished with value: 0.8806556312885426 and parameters: {'C': 0.2931630479904749, 'Class_weight': 'balanced'}. Best is trial 2 with value: 0.8857189224277832.
[I 2026-06-24 19:35:50,201] Trial 11 finished with value: 0.7969165855241804 and parameters: {'C': 0.02974122152593529, 'Class_weight': None}. Best is trial 2 with value: 0.8857189224277832.
[I 2026-06-24 19:35:50,202] Trial 14 finished with value: 0.8375527426160339 and parameters: {'C': 0.05145403021232699, 'Class_weight': 'balanced'}. Best is trial 2 with value: 0.8857189224277832.
[I 2026-06-24 19:35:50,202] Trial 12 finished with value: 0.8933138591366439 and parameters: {'C': 0.9168408011523

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
3,3,0.893314,2026-06-24 19:35:50.108121,2026-06-24 19:35:50.203116,0 days 00:00:00.094995,0.916338,balanced,COMPLETE
10,10,0.893314,2026-06-24 19:35:50.116627,2026-06-24 19:35:50.204117,0 days 00:00:00.087490,0.897863,None,COMPLETE
8,8,0.893314,2026-06-24 19:35:50.114627,2026-06-24 19:35:50.204117,0 days 00:00:00.089490,0.871702,None,COMPLETE
12,12,0.893314,2026-06-24 19:35:50.120793,2026-06-24 19:35:50.202117,0 days 00:00:00.081324,0.916841,balanced,COMPLETE
5,5,0.890782,2026-06-24 19:35:50.110628,2026-06-24 19:35:50.203116,0 days 00:00:00.092488,0.821668,balanced,COMPLETE


In [38]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_f1_optuna_results)

[I 2026-06-24 19:35:50,235] A new study created in memory with name: no-name-c6b7cefb-10c7-4242-81c4-4db515b589d2
[I 2026-06-24 19:35:50,251] Trial 0 finished with value: 0.9236402061329537 and parameters: {'C': 0.26470429767396964, 'Class_weight': None}. Best is trial 0 with value: 0.9236402061329537.
[I 2026-06-24 19:35:50,271] Trial 1 finished with value: 0.9189235607225996 and parameters: {'C': 0.13515677287381883, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.9236402061329537.
[I 2026-06-24 19:35:50,298] Trial 2 finished with value: 0.9252932469235157 and parameters: {'C': 0.3681540561271384, 'Class_weight': 'balanced'}. Best is trial 2 with value: 0.9252932469235157.
[I 2026-06-24 19:35:50,314] Trial 3 finished with value: 0.9304212171893221 and parameters: {'C': 0.6723940063747261, 'Class_weight': None}. Best is trial 3 with value: 0.9304212171893221.
[I 2026-06-24 19:35:50,329] Trial 4 finished with value: 0.9320368539392762 and parameters: {'C': 0.8002068029038181

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
11,11,0.935037,2026-06-24 19:35:50.424098,2026-06-24 19:35:50.439543,0 days 00:00:00.015445,0.994427,balanced,COMPLETE
10,10,0.933644,2026-06-24 19:35:50.408956,2026-06-24 19:35:50.424098,0 days 00:00:00.015142,0.964795,balanced,COMPLETE
12,12,0.933644,2026-06-24 19:35:50.439543,2026-06-24 19:35:50.467171,0 days 00:00:00.027628,0.977202,balanced,COMPLETE
13,13,0.933644,2026-06-24 19:35:50.467171,2026-06-24 19:35:50.482206,0 days 00:00:00.015035,0.985451,balanced,COMPLETE
8,8,0.933469,2026-06-24 19:35:50.376974,2026-06-24 19:35:50.393088,0 days 00:00:00.016114,0.875933,balanced,COMPLETE


##### Tryout best params

In [39]:
lasso_recall_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.248927, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (Recall Optimized)')

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


              precision    recall  f1-score   support

           0       0.94      0.99      0.97        99
           1       0.99      0.94      0.96        98

    accuracy                           0.96       197
   macro avg       0.97      0.96      0.96       197
weighted avg       0.97      0.96      0.96       197



In [40]:
lasso_f1_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.496023, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.95      0.99      0.97        99
           1       0.99      0.95      0.97        98

    accuracy                           0.97       197
   macro avg       0.97      0.97      0.97       197
weighted avg       0.97      0.97      0.97       197



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Conclusions

In [41]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logistic Regression            0              0.958333  0.929293   
                                        1              0.930693  0.959184   
                                        accuracy       0.944162  0.944162   
                                        macro avg      0.944513  0.944238   
                                        weighted avg   0.944583  0.944162   
Baseline Ridge Classifier               0              0.890909  0.989899   
                                        1              0.988506  0.877551   
                                        accuracy       0.934010  0.934010   
                                        macro avg      0.939707  0.933725   
                                        weighted avg   0.939460  0.934010   
Baseline L1 (Lasso) Logistic Regression 0              0.949495  0.949495   
                                        1              0.948980  0.948980   
                                        accuracy       0.949239  0.949239   
                                        macro avg      0.949237  0.949237   
                                        weighted avg   0.949239  0.949239   
Logistic Regression (Recall Optimized)  0              0.950980  0.979798   
                                        1              0.978947  0.948980   
                                        accuracy       0.964467  0.964467   
                                        macro avg      0.964964  0.964389   
                                        weighted avg   0.964893  0.964467   
Logistic Regression (F1 Optimized)      0              0.950980  0.979798   
                                        1              0.978947  0.948980   
                                        accuracy       0.964467  0.964467   
                                        macro avg      0.964964  0.964389   
                                        weighted avg   0.964893  0.964467   
Ridge Classifier (Recall Optimized)     0              0.890909  0.989899   
                                        1              0.988506  0.877551   
                                        accuracy       0.934010  0.934010   
                                        macro avg      0.939707  0.933725   
                                        weighted avg   0.939460  0.934010   
Ridge Classifier (F1 Optimized)         0              0.890909  0.989899   
                                        1              0.988506  0.877551   
                                        accuracy       0.934010  0.934010   
                                        macro avg      0.939707  0.933725   
                                        weighted avg   0.939460  0.934010   
Lasso Classifier (Recall Optimized)     0              0.942308  0.989899   
                                        1              0.989247  0.938776   
                                        accuracy       0.964467  0.964467   
                                        macro avg      0.965778  0.964337   
                                        weighted avg   0.965658  0.964467   
Lasso Classifier (F1 Optimized)         0              0.951456  0.989899   
                                        1              0.989362  0.948980   
                                        accuracy       0.969543  0.969543   
                                        macro avg      0.970409  0.969439   
                                        weighted avg   0.970313  0.969543   

                                                      f1-score     support  
Baseline Logistic Regression            0             0.943590   99.000000  
                                        1             0.944724   98.000000  
                                        accuracy      0.944162    0.944162  
                                        macro avg     0.944157  197.000000  
                                        weighted avg  0.944154  197.000000  
Baseline Ridge Classifier               0        

The best model is the Lasso with macro avg precision 0.965778 recall 0.964337 and f1 0.964434. All models performed very well though on this dataset. Lasso could have been better than log reg due to multicollinearity

In [42]:
import numpy as np

pd.set_option('display.max_rows', None)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Ridge_Coef': ridge_f1_optimized_model.coef_,
})

# Sort features by the absolute size of their coefficients
coef_df['Abs_Coefficient'] = np.abs(coef_df['Ridge_Coef'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)
coef_df

,Feature,Ridge_Coef,Abs_Coefficient
7,V7,0.521043,0.521043
14,V14,-0.510886,0.510886
4,V4,0.446034,0.446034
1,V1,-0.331711,0.331711
10,V10,-0.212063,0.212063
18,V18,0.200497,0.200497
8,V8,-0.178609,0.178609
17,V17,-0.131455,0.131455
3,V3,-0.119278,0.119278
23,V23,-0.112265,0.112265


In [43]:
len(coef_df[coef_df['Abs_Coefficient'] == 0])

0

We can see 0 features coefs get shrunk to 0 by the Ridge model. All features must be at least slightly important

In [44]:
coefficients = lasso_f1_optimized_model.coef_[0]

# 3. Create a DataFrame to map features to their coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})

# 4. VIEW RETAINED FEATURES (Coefficient is NOT 0)
selected_features = coef_df[coef_df['Coefficient'] != 0]
print("Features Selected by Lasso:")
print(selected_features)

# 5. VIEW REJECTED FEATURES (Coefficient is exactly 0)
dropped_features = coef_df[coef_df['Coefficient'] == 0]
print("\nFeatures Ignored by Lasso:")
print(dropped_features['Feature'].tolist())

Features Selected by Lasso:
   Feature  Coefficient
0     Time    -0.054337
3       V3    -0.004034
4       V4     2.157277
5       V5     0.137784
6       V6    -0.116894
8       V8    -0.891354
9       V9    -0.192883
10     V10    -1.167566
11     V11     0.474983
12     V12    -1.297026
13     V13    -0.316520
14     V14    -2.410177
18     V18     0.177864
20     V20    -0.201561
22     V22     0.410333
23     V23    -0.265760
25     V25    -0.033709
26     V26    -0.040076
27     V27    -0.242138
28     V28     0.277115
29  Amount     0.384781

Features Ignored by Lasso:
['V1', 'V2', 'V7', 'V15', 'V16', 'V17', 'V19', 'V21', 'V24']


We can see Lasso also removed 9 features during feature selection